# PCA feature analysis — how much structure is in the 6,693 features?

A dimensionality study of the feature space: how many principal components carry the variance (for all samples, and per class), which original features load on the top components, what the data looks like in 2D, and finally whether a DNN retrained on **91 PCA components** still detects well.

*Consolidated from `correlation_malware.ipynb` (originally in `a_malware2`). The original's own markdown explanations are kept; an unrelated Androguard tail at the end of the original was removed. Outputs cleared.*

## Step 1 — Setup and data

In [ ]:
import time
import os.path as path
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import os
import random

torch.manual_seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

!git clone https://github.com/mostafa-ja/malware_detection.git

In [ ]:
import gdown
# our sharable address : https://drive.google.com/file/d/1-MPYbRTFn_zH3rybfAYihHuIjq2XSiI5/view?usp=sharing
download_link = 'https://drive.google.com/uc?id=1-MPYbRTFn_zH3rybfAYihHuIjq2XSiI5'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1--gjFTkAJUW74RXWE4anJ5wssV7K_VAo'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1-2NnBk3YKauGTGK8DfQHBxrLrYyTxNsp'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1-4PhQGYHpMXrCcxcgMadgPPA4S-9yluX'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1JmP8VxTUU3Yi7dle247PKlgrS6WS91x8'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=16jkBpZh17DFDhgm3wzr1gDIHKSFq0AyL'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
!cp /content/train_dataset.pt /content/malware_detection/datasets
!cp /content/validation_dataset.pt /content/malware_detection/datasets
!cp /content/test_dataset.pt /content/malware_detection/datasets

In [ ]:
import torch
import torch.utils.data as data

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, samples):
        self.samples = [(torch.tensor(data, dtype=torch.float32), label) for data, label in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

In [ ]:
class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dense_hidden_units=[200, 200], dropout=0.6, smooth=False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.name = name

        self.dropout = dropout
        self.smooth = smooth
        self.dense_hidden_units = dense_hidden_units

        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))

        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu

        self.model_save_path = path.join('/content/malware_detection/save/drebin/md_dnn'+ '_' + self.name,'model.pth')

    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits

    def customize_loss(self, logits, gt_labels, representation = None, mini_batch_idx=None):
        return F.cross_entropy(logits, gt_labels)

    # for predict pertubed malicious and create a y_true same size for theam
    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

    def inference(self, test_data_producer):
        confidences = []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer: # for large dataset we have to consider test dataset with batches
                x, y = x.double().to(device), y.long().to(device)
                logits = self.forward(x)
                confidences.append(F.softmax(logits, dim=-1))
                gt_labels.append(y)
        confidences = torch.vstack(confidences) #[[1,2,3],[4,5,6]] > below each other
        gt_labels = torch.cat(gt_labels, dim=0) #[[1,2,3],[4,5,6]] > [1,2,3,4,5,6]
        return confidences, gt_labels

In [ ]:
train_dataset = torch.load('/content/malware_detection/datasets/train_dataset.pt')
train_Loader = data.DataLoader(train_dataset, batch_size=10000, shuffle=False)
validation_dataset = torch.load('/content/malware_detection/datasets/validation_dataset.pt')
validation_Loader = data.DataLoader(validation_dataset, batch_size=10000, shuffle=False)
test_dataset = torch.load('/content/malware_detection/datasets/test_dataset.pt')
test_Loader = data.DataLoader(test_dataset, batch_size=10000, shuffle=False)

In [ ]:
X1,y1 = next(iter(train_Loader))
X2,y2 = next(iter(validation_Loader))
X3,y3 = next(iter(test_Loader))

In [ ]:
X = torch.cat((X1, X2, X3), dim=0)
y = torch.cat((y1, y2, y3), dim=0)
mal_x = X[y==1]
ben_x = X[y==0]

In [ ]:
print(X.shape)
print(mal_x.shape)
print(ben_x.shape)

## Step 2 — PCA on all samples: the variance profile

In [ ]:
from sklearn.decomposition import PCA

components = None
pca = PCA(n_components = components)
# perform PCA on the scaled data
pca.fit(X)

In [ ]:
print(pca.components_.shape) # first row , each one a new dimension, sorted by importance
# pca.components_[0] # the weight of each feature in the first pca dimension
print(pca.components_)

In [ ]:
# print the explained variances
print("Variances (Percentage):")
print(pca.explained_variance_ratio_ [:20]* 100)

In [ ]:
print(pca.explained_variance_ratio_ [:20].cumsum()* 100)

In [ ]:
import matplotlib.pyplot as plt
component = len(pca.explained_variance_ratio_)
plt.plot(range(1,component+1), pca.explained_variance_ratio_.cumsum()*100);
plt.xlabel('Number of components');
plt.ylabel('Explained variance (%)');

In [ ]:
import matplotlib.pyplot as plt
component = len(pca.explained_variance_ratio_[:500])
plt.plot(range(1,component+1), pca.explained_variance_ratio_[:500].cumsum()*100);
plt.xlabel('Number of components');
plt.ylabel('Explained variance (%)');

## Step 3 — PCA per class (benign vs malicious)

In [ ]:
pca_ben = PCA(n_components = components)
# perform PCA on the scaled data
pca_ben.fit(ben_x)

print(pca_ben.explained_variance_ratio_ [:20]* 100)

In [ ]:
print(pca_ben.explained_variance_ratio_ [:20].cumsum()* 100)

In [ ]:
component = len(pca_ben.explained_variance_ratio_)
plt.plot(range(1,component+1), pca_ben.explained_variance_ratio_.cumsum()*100);
plt.xlabel('Number of components');
plt.ylabel('Explained variance (%)');

In [ ]:
component = len(pca_ben.explained_variance_ratio_[:100])
plt.plot(range(1,component+1), pca_ben.explained_variance_ratio_[:100].cumsum()*100);
plt.xlabel('Number of components');
plt.ylabel('Explained variance (%)');

In [ ]:
pca_mal = PCA(n_components = components)
# perform PCA on the scaled data
pca_mal.fit(mal_x)

print(pca_mal.explained_variance_ratio_ [:20]* 100)

In [ ]:
print(pca_mal.explained_variance_ratio_ [:20].cumsum()* 100)

In [ ]:
component = len(pca_mal.explained_variance_ratio_)
plt.plot(range(1,component+1), pca_mal.explained_variance_ratio_.cumsum()*100);
plt.xlabel('Number of components');
plt.ylabel('Explained variance (%)');

In [ ]:
print("Cumulative Variances (Percentage):")
print(np.cumsum(pca.explained_variance_ratio_ * 100))
components = len(pca.explained_variance_ratio_)
print(f'Number of components: {components}')
# Make the scree plot
plt.plot(range(1, components + 1), np.cumsum(pca.explained_variance_ratio_ * 100));
plt.xlabel("Number of components");
plt.ylabel("Explained variance (%)");

## Step 4 — Keeping 70% of the variance, and the features behind the components

The scree plot makes it easy for you to visualize the number of components that are needed to capture the various amount of variability in the data:

Let’s now apply PCA to find the desired number of components based on the desired explained variance, say 70%:

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components = 0.70)
pca.fit(X)
print("Cumulative Variances (Percentage):")
print(np.cumsum(pca.explained_variance_ratio_ * 100))
components = len(pca.explained_variance_ratio_)
print(f'Number of components: {components}')
# Make the scree plot
plt.plot(range(1, components + 1), np.cumsum(pca.explained_variance_ratio_ * 100));
plt.xlabel("Number of components");
plt.ylabel("Explained variance (%)");

You can also find out the importance of each feature that contributes to each of the components using the components_ attribute of the pca object:

In [ ]:
print(pca.components_.shape)
pca_components = abs(pca.components_)
print(pca_components)

In [ ]:
import pickle

# Open and load the Pickle file
with open('/content/malware_detection/dataset/drebin/data.vocab', 'rb') as file:
    features = pickle.load(file)

# Access and use the loaded data
print(len(features))  # Example: Print the loaded data

In [ ]:
features[:10]

In [ ]:
import pandas as pd

index = ['Component'+str(i+1) for i in range(pca.components_.shape[0])]
columns = features

df_components = pd.DataFrame(pca.components_,index=index, columns=columns)
df_components.head()

In [ ]:
np.sort(pca.components_[0])

For curiosity, let’s print out the top 4 features that contributes the most to each of the components:

In [ ]:
print('Top 4 most important features in each component')
print('===============================================')
for row in range(pca_components.shape[0]):

    # sort the indices in descending order
    indices = np.argsort(pca_components[row])[::-1][:4]  # [::-1] : reverse the list from max to min

    # print the top 4 feature names
    print(f'Component {row}: {df_components.columns[indices].to_list()}')

## Step 5 — The same for the malicious class alone

In [ ]:
pca_mal = PCA(n_components = 0.70)
pca_mal.fit(mal_x)
print("Cumulative Variances (Percentage):")
print(np.cumsum(pca_mal.explained_variance_ratio_ * 100))
components = len(pca_mal.explained_variance_ratio_)
print(f'Number of components: {components}')
# Make the scree plot
plt.plot(range(1, components + 1), np.cumsum(pca_mal.explained_variance_ratio_ * 100));
plt.xlabel("Number of components");
plt.ylabel("Explained variance (%)");

In [ ]:
index_mal = ['Component'+str(i+1) for i in range(pca_mal.components_.shape[0])]
columns = features

df_components_mal = pd.DataFrame(pca_mal.components_,index=index_mal, columns=columns)
df_components_mal.head()

In [ ]:
print('Top 4 most important features in each component')
print('===============================================')
pca_components_mal = abs(pca_mal.components_)
for row in range(pca_components_mal.shape[0]):

    # sort the indices in descending order
    indices = np.argsort(pca_components_mal[row])[::-1][:4]  # [::-1] : reverse the list from max to min

    # print the top 4 feature names
    print(f'Component {row}: {df_components_mal.columns[indices].to_list()}')

## Step 6 — Down to 2D: PCA scatter and t-SNE

#PCA Visualization

As we've noticed before it is difficult to visualize high dimensional data, we can use PCA to find the first two principal components, and visualize the data in this new, two-dimensional space, with a single scatter-plot. Before we do this though, we'll need to scale our data so that each feature has a single unit variance.

PCA with Scikit Learn uses a very similar process to other preprocessing functions that come with SciKit Learn. We instantiate a PCA object, find the principal components using the fit method, then apply the rotation and dimensionality reduction by calling transform().

We can also specify how many components we want to keep when creating the PCA object.

In [ ]:
pca = PCA(n_components=2)
pca.fit(X)

Now we can transform this data to its first 2 principal components.



In [ ]:
x_pca = pca.transform(X)
x_pca.shape

In [ ]:
X.shape

Great! We've reduced 6693 dimensions to just 2! Let's plot these two dimensions out!



In [ ]:
y

In [ ]:
plt.figure(figsize=(8,6))
plt.grid()
plt.scatter(x_pca[:,0],x_pca[:,1],c=y)
plt.xlabel('First principal component')
plt.ylabel('Second Principal Component')

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np


# Create a t-SNE instance specifying the number of components (2 for 2D)
tsne = TSNE(n_components=2)

# Fit the data and transform it to 2D
data_2d = tsne.fit_transform(X)

# Plot the 2D representation of the data
plt.scatter(data_2d[:, 0], data_2d[:, 1],c=y)
plt.title('t-SNE Visualization of High-Dimensional Data')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

## Step 7 — Retrain the DNN on 91 PCA components

# Training
train the model with pca components

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
pca = PCA(n_components = 0.85)
pca.fit(X)
print("Cumulative Variances (Percentage):")
print(np.cumsum(pca.explained_variance_ratio_ * 100))
components = len(pca.explained_variance_ratio_)
print(f'Number of components: {components}')
# Make the scree plot
plt.plot(range(1, components + 1), np.cumsum(pca.explained_variance_ratio_ * 100));
plt.xlabel("Number of components");
plt.ylabel("Explained variance (%)");

In [ ]:
train_Loader = data.DataLoader(train_dataset, batch_size=128, shuffle=False)

validation_Loader = data.DataLoader(validation_dataset, batch_size=10000, shuffle=False)

test_Loader = data.DataLoader(test_dataset, batch_size=10000, shuffle=False)

In [ ]:
model = MalwareDetectionDNN(input_size=91,n_classes=2,dense_hidden_units=[46, 20], dropout=0.6)
model = model.to(device)
model

In [ ]:
from torchsummary import summary

summary(model, input_size=(91,))

In [ ]:
lr =0.005
weight_decay = 0.
epochs = 50
dropout=0.6

optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()
best_avg_acc = 0.
best_epoch = 0
total_time = 0.
nbatches = len(train_Loader)
for i in range(epochs):
    model.train()
    losses, accuracies = [], []
    for idx_batch, (x_train, y_train) in enumerate(train_Loader):
        x_train = pca.transform(x_train)
        x_train = torch.tensor(x_train)
        x_train = x_train.to(device)
        y_train = y_train.long().to(device)
        start_time = time.time()
        optimizer.zero_grad()
        logits = model.forward(x_train.float())
        loss_train = loss_fn(logits, y_train)
        loss_train.backward()
        optimizer.step()
        total_time = total_time + time.time() - start_time
        acc_train = (logits.argmax(1) == y_train).sum().item()
        acc_train /= x_train.size()[0]
        mins, secs = int(total_time / 60), int(total_time % 60)
        losses.append(loss_train.item())
        accuracies.append(acc_train)

        if True:
            print(f'Mini batch: {i * nbatches + idx_batch + 1}/{epochs * nbatches} | training time in {mins:.0f} minutes, {secs} seconds.')
            print(f'Training loss (batch level): {losses[-1]:.4f} | Train accuracy: {acc_train * 100:.2f}')

    model.eval()
    avg_acc_val = []
    with torch.no_grad():
        for x_val, y_val in validation_Loader:
            x_val = pca.transform(x_val)
            x_val = torch.tensor(x_val)
            x_val = x_val.to(device)
            y_val = y_val.long().to(device)
            logits = model.forward(x_val.float())
            acc_val = (logits.argmax(1) == y_val).sum().item()
            acc_val /= x_val.size()[0]
            avg_acc_val.append(acc_val)
        avg_acc_val = np.mean(avg_acc_val)

    if avg_acc_val >= best_avg_acc:
        best_avg_acc = avg_acc_val
        best_epoch = i
        torch.save(model.state_dict(), 'model.pth')

    if True:
        print(f'Training loss (epoch level): {np.mean(losses):.4f} | Train accuracy: {np.mean(accuracies) * 100:.2f}')
        print(f'Validation accuracy: {avg_acc_val * 100:.2f} | The best validation accuracy: {best_avg_acc * 100:.2f} at epoch: {best_epoch}')

In [ ]:
model.load_state_dict(torch.load('/content/model.pth'))

model.eval()
with torch.no_grad():
    for x, y in test_Loader:
        x = pca.transform(x)
        x = torch.tensor(x)
        x = x.to(device)
        logits = model.forward(x.float())
        confidences = F.softmax(logits, dim=-1)


y_pred = confidences.argmax(1).cpu().numpy()
y_true = y.cpu().numpy()

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
accuracy = accuracy_score(y_true, y_pred)
b_accuracy = balanced_accuracy_score(y_true, y_pred)
print("The accuracy on the test dataset is {:.5f}%".format(accuracy * 100))
print("The balanced accuracy on the test dataset is {:.5f}%".format(b_accuracy * 100))

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / float(tn + fp)
fnr = fn / float(tp + fn)
f1 = f1_score(y_true, y_pred, average='binary')

print("Other evaluation metrics we may need:")
print("False Negative Rate (FNR) is {:.5f}%, False Positive Rate (FPR) is {:.5f}%, F1 score is {:.5f}%".format(fnr * 100, fpr * 100, f1 * 100))

In [ ]:
x_val.shape

In [ ]:
x_val[0]